# Master evaluation

Scores every run under `runs/` the same way and writes its `scores.json`. The decodability floors go to `runs/_baselines/`. A run already scored at the current eval version, with every block the settings ask for, is skipped. The scorer is `pim/scoring/`; no metrics are defined here.

In [ ]:
import os
import sys
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pim").is_dir())
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from pim.scoring import print_summaries, scan_runs, score_all, score_all_baselines  # noqa: E402

RUNS = scan_runs()
print(f"{'run':<28} {'arch':<22} {'instance':<16} scored")
for r in RUNS:
    print(f"{r['id']:<28} {r['arch']:<22} {r['instance']:<16} {r['scored']}")

## Settings

Every knob of the evaluation. Categorical-target probes and their floors are read from the cache, never fitted here: fit them first with `scripts/fit_probes.py` at `rw_cat_probe_seqs` sequences and `rw_cat_probe_epochs` epochs (its defaults).

In [ ]:
ALPHA_REG = {"pi": (0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0, 35.0, 60.0, 100.0, 175.0),
             "gs": (0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7)}
ALPHA_CAT = {"pi": (0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0, 35.0, 60.0, 100.0),
             "gs": (0.05, 0.1, 0.2, 0.35, 0.7, 1.5)}
GS_LAYERS = (0, 2, 4, 6, 8)   # GS start points (residual points)
APP_FAC = ("appearance-fac",)
N_RAY = ("rayworld/128-ray", "rayworld/16-ray", "rayworld/8-ray", "rayworld/5-ray")

SETTINGS = {
    # probes: Rayworld sequences from probe_120k, Othello games from the probe split
    "rw_probe_seqs": 30_000,
    "oth_probe_games": 20_000,
    # categorical-target probes: sequences from probe_250k and epochs
    "rw_cat_probe_seqs": 200_000,
    "rw_cat_probe_epochs": 50,
    # Rayworld edit bench and editors
    "rw_bench_n": 1000,
    "rw_target": "full",
    "rw_edit_dims": ("all",),
    "rw_bases": ("frustum", "cartesian"),   # frustum first: categorical probes are keyed under the first basis
    "rw_bases_by_instance": {"rayworld/obs5": ("cartesian",)},
    "rw_alpha_pi": ALPHA_REG["pi"], "rw_alpha_gs": ALPHA_REG["gs"],
    "rw_grid_alpha_pi": ALPHA_CAT["pi"], "rw_grid_alpha_gs": ALPHA_CAT["gs"],
    "gs_layers": GS_LAYERS,
    "rw_gs_steps": 100,
    "rw_gs_beta": 0.2,
    # extra probe targets per run id (a seed replicate is listed under its own id), each its own block
    "rw_extra_targets": {
        "rayworld/standard": APP_FAC, "rayworld/blink": APP_FAC,   # for the qualitative figures; no floors, no IM
        "rayworld/8-ray": ("appearance-fac", "appearance", "grid-6x5", "grid-10x3", "grid-16x8", "pos@appearance"),
        "rayworld/8-ray__seed0": APP_FAC, "rayworld/8-ray__seed1": APP_FAC, "rayworld/8-ray__seed2": APP_FAC,
        "rayworld/8-ray-tokens": APP_FAC,
        "rayworld/128-ray": APP_FAC,
        "rayworld/128-ray__seed0": APP_FAC, "rayworld/128-ray__seed1": APP_FAC, "rayworld/128-ray__seed2": APP_FAC,
        "rayworld/16-ray": APP_FAC,
        "rayworld/16-ray__seed0": APP_FAC, "rayworld/16-ray__seed1": APP_FAC, "rayworld/16-ray__seed2": APP_FAC,
        "rayworld/5-ray": APP_FAC,
        "rayworld/5-ray__seed0": APP_FAC, "rayworld/5-ray__seed1": APP_FAC, "rayworld/5-ray__seed2": APP_FAC,
    },
    # extra targets that get decodability floors, on these instances
    "rw_floor_targets": {"instances": N_RAY, "targets": ("appearance-fac", "appearance", "pos@appearance")},
    # categorical blocks that get an IM arm
    "rw_cat_im": {"instances": N_RAY,
                  "targets": ("appearance-fac", "appearance", "grid-6x5", "grid-10x3", "grid-16x8")},
    # Othello edit bench and editors
    "oth_alpha_pi": ALPHA_CAT["pi"], "oth_alpha_gs": ALPHA_CAT["gs"],
    "oth_gs_layers": GS_LAYERS,
    "oth_gs_steps": 100,
    "oth_gs_beta": 0.2,
    "oth_gates_games": 10_000,
}

## Decodability floors

Observation and random-init floors per environment, instance and architecture; only what a `baselines.json` lacks is fitted.

In [ ]:
score_all_baselines(RUNS, SETTINGS)

## Score

A run without a current `scores.json` is scored in full; a current one that lacks a block gets only that block added. A full rescore drops the `prediction` block, which `scripts/score_prediction.py` adds.

In [ ]:
score_all(RUNS, SETTINGS)

## Summaries

Each editor's top arm before the fidelity cutoff, with its fidelity ratio (`ratio`, 1 minus Edit Fidelity). The table notebooks apply the paper's selection rule.

In [ ]:
print_summaries(RUNS)